# Wreckognise — GPU training on Colab

Trains the side-scan sonar detector on **SCTD + AI4Shipwrecks** and exports
ONNX weights plus a measured `metrics.json` ready to drop into `backend/models/`.

On a free T4 this takes **~15–25 minutes**. The same run takes ~26 hours on a laptop CPU.

---
### Before you start
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save.**

If you skip that, everything still runs but on CPU, and you gain nothing.


## 1. Confirm the GPU is actually attached


In [ ]:
import torch, subprocess

print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
      or 'no nvidia-smi — you are on CPU')
print('torch sees CUDA:', torch.cuda.is_available())

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.'
)


## 2. Install


In [ ]:
!pip -q install ultralytics onnx onnxslim
import ultralytics; print(ultralytics.__version__)


## 3. Fetch SCTD

357 real side-scan images with Pascal VOC boxes (ship / aircraft / human).
Public repo, clones directly.


In [ ]:
!git clone -q --depth 1 https://github.com/MingqiangNing/SCTD.git /content/sctd_repo
!cd /content/sctd_repo && unzip -q -o SCTD.zip -d /content/sctd

from pathlib import Path
imgs = list(Path('/content/sctd').rglob('*.jpg'))
xmls = list(Path('/content/sctd').rglob('*.xml'))
print(f'SCTD: {len(imgs)} images, {len(xmls)} annotations')


## 4. Upload AI4Shipwrecks

Not scriptable — Deep Blue is behind bot protection. Download it once from
<https://deepblue.lib.umich.edu/data/concern/data_sets/8623hz41x>, then run this cell
and pick the zip.

**Skip this cell** to train on SCTD alone (you'll get ~0.84 in-distribution but poor transfer).


In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()          # choose AI4Shipwrecks.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/ai4sw')

root = '/content/ai4sw/AI4Shipwrecks'
print('train images:', len(os.listdir(f'{root}/train/images')))
print('test images :', len(os.listdir(f'{root}/test/images')))


## 5. Build the combined YOLO dataset

Two things this cell does that matter:

* **Tiles AI4Shipwrecks** into 512 px patches. Its images are 5579×1728 full swaths where a
  wreck is ~2.5% of the frame — squashed to 640 px a wreck becomes ~15 px and is unlearnable.
  Tiling also matches how the API runs inference.
* **Keeps empty-seabed tiles** as negatives. They are what teaches the model not to fire on
  ripple texture, which is the main source of false positives.

Splits follow each dataset's own boundary, so no tile of a wreck leaks between train and val.


In [ ]:
import random, shutil, xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
import cv2, numpy as np

CLASSES = ['aircraft', 'human', 'ship']
SHIP_ID = CLASSES.index('ship')
OUT = Path('/content/dataset')
TILE, STRIDE = 512, 384
MIN_BLOB_PX, MIN_BOX_PX, NEG_RATIO = 60, 12, 0.6
rng = random.Random(1337)

if OUT.exists(): shutil.rmtree(OUT)
for s in ('train','val'):
    (OUT/'images'/s).mkdir(parents=True, exist_ok=True)
    (OUT/'labels'/s).mkdir(parents=True, exist_ok=True)

# ---------- SCTD: Pascal VOC -> YOLO, stratified 80/20 ----------
def parse_voc(x):
    try: root = ET.parse(x).getroot()
    except ET.ParseError: return None, None, []
    sz = root.find('size')
    if sz is None: return None, None, []
    w, h = int(float(sz.findtext('width'))), int(float(sz.findtext('height')))
    out = []
    for o in root.findall('object'):
        n = (o.findtext('name') or '').strip().lower()
        bb = o.find('bndbox')
        if not n or bb is None or n not in CLASSES: continue
        x0,y0 = float(bb.findtext('xmin')), float(bb.findtext('ymin'))
        x1,y1 = float(bb.findtext('xmax')), float(bb.findtext('ymax'))
        x0,x1 = sorted((max(0,x0), min(w,x1))); y0,y1 = sorted((max(0,y0), min(h,y1)))
        if x1-x0 >= 2 and y1-y0 >= 2: out.append((n,x0,y0,x1,y1))
    return w, h, out

sctd_imgs = {p.stem: p for p in Path('/content/sctd').rglob('*.jpg')}
sctd_xmls = {p.stem: p for p in Path('/content/sctd').rglob('*.xml')}
samples = []
for k in sorted(sctd_imgs.keys() & sctd_xmls.keys()):
    w,h,b = parse_voc(sctd_xmls[k])
    if w and h and b: samples.append((k,w,h,b))

by_cls = defaultdict(list)
for e in samples:
    by_cls[Counter(x[0] for x in e[3]).most_common(1)[0][0]].append(e)
train, val = [], []
for _, entries in by_cls.items():
    rng.shuffle(entries)
    cut = max(1, round(len(entries)*0.2))
    val += entries[:cut]; train += entries[cut:]

for split, entries in (('train',train), ('val',val)):
    for k,w,h,boxes in entries:
        shutil.copy2(sctd_imgs[k], OUT/'images'/split/f'{k}.jpg')
        lines = [f'{CLASSES.index(n)} {((x0+x1)/2)/w:.6f} {((y0+y1)/2)/h:.6f} {(x1-x0)/w:.6f} {(y1-y0)/h:.6f}'
                 for n,x0,y0,x1,y1 in boxes]
        (OUT/'labels'/split/f'{k}.txt').write_text('\n'.join(lines))
print(f'SCTD -> train {len(train)}, val {len(val)}')

# ---------- AI4Shipwrecks: masks -> tiled boxes ----------
ai4 = Path('/content/ai4sw/AI4Shipwrecks')
if ai4.exists():
    def mask_boxes(m):
        n,_,st,_ = cv2.connectedComponentsWithStats((m>0).astype(np.uint8), 8)
        return [tuple(int(v) for v in st[i][:4]) for i in range(1,n)
                if st[i][4] >= MIN_BLOB_PX and st[i][2] >= 4 and st[i][3] >= 4]

    for src, dstsplit in (('train','train'), ('test','val')):
        pos, neg = [], []
        for ip in sorted((ai4/src/'images').glob('*.png')):
            im = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
            mk = cv2.imread(str(ai4/src/'labels'/ip.name), cv2.IMREAD_GRAYSCALE)
            if im is None or mk is None: continue
            if mk.shape != im.shape:
                mk = cv2.resize(mk, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
            boxes = mask_boxes(mk); H, W = im.shape
            for top in range(0, max(1,H-TILE+1), STRIDE):
                for left in range(0, max(1,W-TILE+1), STRIDE):
                    bot, right = min(top+TILE,H), min(left+TILE,W)
                    if bot-top < TILE//2 or right-left < TILE//2: continue
                    local = []
                    for bx,by,bw,bh in boxes:
                        ix0,iy0 = max(bx,left), max(by,top)
                        ix1,iy1 = min(bx+bw,right), min(by+bh,bot)
                        if ix1-ix0 < MIN_BOX_PX or iy1-iy0 < MIN_BOX_PX: continue
                        if (ix1-ix0)*(iy1-iy0) < 0.35*bw*bh: continue
                        local.append((ix0-left, iy0-top, ix1-ix0, iy1-iy0))
                    rec = (ip, left, top, right-left, bot-top, local)
                    (pos if local else neg).append(rec)
        rng.shuffle(neg); neg = neg[:int(len(pos)*NEG_RATIO)]
        for i,(ip,left,top,tw,th,boxes) in enumerate(pos+neg):
            im = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
            stem = f'ai4_{ip.stem}_{left}_{top}_{i}'
            cv2.imwrite(str(OUT/'images'/dstsplit/f'{stem}.jpg'), im[top:top+th, left:left+tw])
            (OUT/'labels'/dstsplit/f'{stem}.txt').write_text('\n'.join(
                f'{SHIP_ID} {(bx+bw/2)/tw:.6f} {(by+bh/2)/th:.6f} {bw/tw:.6f} {bh/th:.6f}'
                for bx,by,bw,bh in boxes))
        print(f'AI4SW {src} -> {dstsplit}: {len(pos)} positive + {len(neg)} empty tiles')
else:
    print('AI4Shipwrecks not uploaded — training on SCTD alone')

(OUT/'data.yaml').write_text(
    f'path: {OUT}\ntrain: images/train\nval: images/val\n\nnames:\n'
    + ''.join(f'  {i}: {c}\n' for i,c in enumerate(CLASSES)))
print('TOTAL train:', len(list((OUT/'images'/'train').glob('*.jpg'))),
      '| val:', len(list((OUT/'images'/'val').glob('*.jpg'))))


## 6. Train

`yolov8s` at 640 px — a bigger model and higher resolution than the laptop run could afford.

Augmentation is sonar-aware. **`flipud=0.0` is deliberate**: a vertical flip puts the acoustic
shadow on the wrong side of the target, destroying the strongest cue the model has. Horizontal
flip is fine — it just swaps port and starboard.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
model.train(
    data='/content/dataset/data.yaml',
    epochs=150, imgsz=640, batch=16, device=0,
    project='/content/runs', name='wreckognise', exist_ok=True,
    patience=40, seed=1337, deterministic=True, plots=True,
    # --- sonar-aware augmentation ---
    fliplr=0.5,        # port/starboard swap: physically valid
    flipud=0.0,        # would invert shadow direction: never
    degrees=5.0, translate=0.10, scale=0.5, shear=0.0, perspective=0.0,
    mosaic=1.0, close_mosaic=15, mixup=0.1,
    hsv_h=0.0, hsv_s=0.0,   # sonar is single-channel intensity
    hsv_v=0.4,              # gain variation between surveys is real
)


## 7. Measure it

Scores the best checkpoint on the combined validation set, and separately on the
AI4Shipwrecks tiles alone — the cross-dataset number that says whether it actually
generalises rather than memorising one corpus.


In [ ]:
import shutil, json
from pathlib import Path
from ultralytics import YOLO

BEST = '/content/runs/wreckognise/weights/best.pt'
res = {}

m = YOLO(BEST).val(data='/content/dataset/data.yaml', imgsz=640, device=0, plots=False)
res['combined'] = dict(map50=float(m.box.map50), map=float(m.box.map),
                       p=float(m.box.mp), r=float(m.box.mr))

# AI4Shipwrecks-only split, for the cross-dataset read
sub = Path('/content/ai4_only')
if sub.exists(): shutil.rmtree(sub)
for s in ('train','val'):
    (sub/'images'/s).mkdir(parents=True, exist_ok=True)
    (sub/'labels'/s).mkdir(parents=True, exist_ok=True)
tiles = sorted(Path('/content/dataset/images/val').glob('ai4_*.jpg'))
for p in tiles:
    shutil.copy2(p, sub/'images'/'val'/p.name)
    lp = Path('/content/dataset/labels/val')/f'{p.stem}.txt'
    if lp.is_file(): shutil.copy2(lp, sub/'labels'/'val'/lp.name)
for p in tiles[:2]:
    shutil.copy2(p, sub/'images'/'train'/p.name)
    shutil.copy2(sub/'labels'/'val'/f'{p.stem}.txt', sub/'labels'/'train'/f'{p.stem}.txt')
(sub/'data.yaml').write_text(
    f'path: {sub}\ntrain: images/train\nval: images/val\n\nnames:\n  0: aircraft\n  1: human\n  2: ship\n')

if tiles:
    m2 = YOLO(BEST).val(data=str(sub/'data.yaml'), imgsz=640, device=0, plots=False)
    res['ai4shipwrecks_only'] = dict(map50=float(m2.box.map50), map=float(m2.box.map),
                                     p=float(m2.box.mp), r=float(m2.box.mr))

print()
for k, v in res.items():
    print(f"  {k:22} mAP50={v['map50']:.4f}  mAP50-95={v['map']:.4f}  P={v['p']:.3f}  R={v['r']:.3f}")


## 8. Export for the backend

Writes `yolov8n-sonar.onnx` (name kept so `onnx_detector.py` finds it) and a `metrics.json`
containing figures this notebook actually measured. The API reads accuracy only from that
file, so a number can never reach the dashboard unless a real evaluation produced it.


In [ ]:
from datetime import datetime, timezone
import json, shutil
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

onnx = YOLO(BEST).export(format='onnx', imgsz=640, simplify=True, opset=12)
out = Path('/content/models'); out.mkdir(exist_ok=True)
shutil.copy2(onnx, out/'yolov8n-sonar.onnx')   # filename the backend expects

c = res['combined']
metrics = {
    'model_version': '2.0.0-combined-colab',
    'map50': round(c['map50'], 4), 'map50_95': round(c['map'], 4),
    'precision': round(c['p'], 4), 'recall': round(c['r'], 4),
    'f1': round(2*c['p']*c['r']/max(c['p']+c['r'],1e-9), 4),
    'cross_dataset': res.get('ai4shipwrecks_only'),
    'validated_on': 'SCTD + AI4Shipwrecks held-out split (split by survey line)',
    'trained_on': 'SCTD 1.0 + AI4Shipwrecks, YOLOv8s @ 640px',
    'imgsz': 640,
    'evaluated_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
}
(out/'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

!cd /content/models && zip -q -r /content/wreckognise_model.zip .
files.download('/content/wreckognise_model.zip')


---
## Installing the result

Unzip into `backend/models/`, replacing both files:

```
backend/models/yolov8n-sonar.onnx
backend/models/metrics.json
```

Then commit and push — Render redeploys automatically. Confirm it took with:

```bash
curl https://wreckognise-api.onrender.com/api/health
```

`detection_engine` should read `yolov8-onnx` and `trained_weights_loaded` should be `true`.

**One thing to check before you ship it:** compare the new `map50` against the current
deployed 0.839. Training on two corpora usually trades in-distribution accuracy for
generalisation, so the headline number may go *down* while the cross-dataset number goes up.
Decide which one you want to stand behind, and say so out loud when presenting.
